In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import fsolve
import pandas as pd

## Define Leg Kinematics

Link lengths from the hexapod leg design

In [2]:
# Leg link lengths (from Leg.h)
HIP_TO_KNEE = 37.0      # a1
KNEE_TO_ANKLE = 63.54   # a2
ANKLE_TO_TIP = 200.0    # a3

MAX_REACH = HIP_TO_KNEE + KNEE_TO_ANKLE + ANKLE_TO_TIP

print(f"Link lengths:")
print(f"  Hip to Knee (a1): {HIP_TO_KNEE} mm")
print(f"  Knee to Ankle (a2): {KNEE_TO_ANKLE} mm")
print(f"  Ankle to Tip (a3): {ANKLE_TO_TIP} mm")
print(f"  Maximum reach: {MAX_REACH:.2f} mm")

Link lengths:
  Hip to Knee (a1): 37.0 mm
  Knee to Ankle (a2): 63.54 mm
  Ankle to Tip (a3): 200.0 mm
  Maximum reach: 300.54 mm


## Implement Inverse Kinematics

Based on the C++ implementation in Kinematics.cpp

In [3]:
# Range limits for each servo (in degrees)
HIP_LIMITS = (-50.0, 50.0)
KNEE_LIMITS = (-95.0, 95.0)
ANKLE_LIMITS = (-125.0, 5.0)

def inverse_kinematics(x, y, z, a1=HIP_TO_KNEE, a2=KNEE_TO_ANKLE, a3=ANKLE_TO_TIP):
    """
    Calculate inverse kinematics for the hexapod leg.
    Includes validation to prevent physically impossible configurations.
    
    Args:
        x, y, z: Target position coordinates
        a1, a2, a3: Link lengths
    
    Returns:
        tuple: (theta1, theta2, theta3) in degrees, or None if unreachable or out of limits
    """
    # Check if target is reachable
    target_dis = np.sqrt(x**2 + y**2 + z**2)
    max_leg_dis = a1 + a2 + a3
    
    if max_leg_dis < target_dis:
        return None  # Out of reach
    
    try:
        # Top view calculation, finding theta1
        theta1 = np.arctan2(y, x)
        r1 = np.sqrt(x**2 + y**2) - a1
        
        # Side view calculation, finding theta2 and theta3
        r2 = z
        phi2 = np.arctan2(r2, r1)
        r3 = np.sqrt(r1**2 + r2**2)
        
        # Clamp values for acos to prevent errors
        val1 = (a3**2 - a2**2 - r3**2) / (-2 * a2 * r3)
        val1 = np.clip(val1, -1, 1)
        phi1 = np.arccos(val1)
        
        theta2 = phi1 + phi2
        
        val2 = (r3**2 - a2**2 - a3**2) / (-2 * a2 * a3)
        val2 = np.clip(val2, -1, 1)
        phi3 = np.arccos(val2)
        
        theta3 = -(np.pi - phi3)
        
        # Convert to degrees
        theta1_deg = np.degrees(theta1)
        theta2_deg = np.degrees(theta2)
        theta3_deg = np.degrees(theta3)
        
        # Check if all angles are within physical limits
        if not (HIP_LIMITS[0] <= theta1_deg <= HIP_LIMITS[1]):
            return None
        if not (KNEE_LIMITS[0] <= theta2_deg <= KNEE_LIMITS[1]):
            return None
        if not (ANKLE_LIMITS[0] <= theta3_deg <= ANKLE_LIMITS[1]):
            return None
        
        return (theta1_deg, theta2_deg, theta3_deg)
    
    except:
        return None

## Grid Sampling - 3D Workspace Analysis

In [4]:
# Define sampling parameters
X_MIN, X_MAX, X_STEP = 0, MAX_REACH, 20.0
Y_MIN, Y_MAX, Y_STEP = -MAX_REACH, MAX_REACH, 20.0
Z_MIN, Z_MAX, Z_STEP = -MAX_REACH, 0.0, 20.0

# Create grid
x_range = np.arange(X_MIN, X_MAX + X_STEP, X_STEP)
y_range = np.arange(Y_MIN, Y_MAX + Y_STEP, Y_STEP)
z_range = np.arange(Z_MIN, Z_MAX + Z_STEP, Z_STEP)

print(f"Grid dimensions:")
print(f"  X: {len(x_range)} points ({X_MIN} to {X_MAX})")
print(f"  Y: {len(y_range)} points ({Y_MIN} to {Y_MAX})")
print(f"  Z: {len(z_range)} points ({Z_MIN} to {Z_MAX})")
print(f"  Total points: {len(x_range) * len(y_range) * len(z_range)}")

Grid dimensions:
  X: 17 points (0 to 300.53999999999996)
  Y: 32 points (-300.53999999999996 to 300.53999999999996)
  Z: 17 points (-300.53999999999996 to 0.0)
  Total points: 9248


In [5]:
# Sample the workspace
reachable_points = []
unreachable_points = []
distances = []

print("Sampling workspace...")

for x in x_range:
    for y in y_range:
        for z in z_range:
            distance = np.sqrt(x**2 + y**2 + z**2)
            result = inverse_kinematics(x, y, z)
            
            if result:
                reachable_points.append((x, y, z))
                distances.append(distance)
            else:
                unreachable_points.append((x, y, z))

# Convert to arrays for easier manipulation
reachable_points = np.array(reachable_points) if reachable_points else np.array([])
unreachable_points = np.array(unreachable_points) if unreachable_points else np.array([])
distances = np.array(distances)

total_samples = len(reachable_points) + len(unreachable_points)

print("\n========== WORKSPACE STATISTICS ==========")
print(f"Total samples: {total_samples}")
print(f"Reachable points: {len(reachable_points)} ({len(reachable_points)/total_samples*100:.1f}%)")
print(f"Unreachable points: {len(unreachable_points)} ({len(unreachable_points)/total_samples*100:.1f}%)")
if len(distances) > 0:
    print(f"Min reach distance: {distances.min():.2f} mm")
    print(f"Max reach distance: {distances.max():.2f} mm")
    print(f"Avg reach distance: {distances.mean():.2f} mm")
print("==========================================")

Sampling workspace...

========== WORKSPACE STATISTICS ==========
Total samples: 9248
Reachable points: 1503 (16.3%)
Unreachable points: 7745 (83.7%)
Min reach distance: 181.65 mm
Max reach distance: 300.47 mm
Avg reach distance: 257.18 mm


## 3D Visualization of Workspace

In [6]:
fig = go.Figure()

# Plot reachable points
if len(reachable_points) > 0:
    fig.add_trace(go.Scatter3d(
        x=reachable_points[:, 0],
        y=reachable_points[:, 1],
        z=reachable_points[:, 2],
        mode='markers',
        name='Reachable',
        marker=dict(
            size=4,
            color='green',
            opacity=0.7,
            line=dict(width=0)
        ),
        text=[f"X: {x:.1f}<br>Y: {y:.1f}<br>Z: {z:.1f}<br>D: {np.sqrt(x**2+y**2+z**2):.2f}mm" 
              for x, y, z in reachable_points],
        hovertemplate='<b>Reachable</b><br>%{text}<extra></extra>'
    ))

# Plot unreachable points
if len(unreachable_points) > 0:
    fig.add_trace(go.Scatter3d(
        x=unreachable_points[:, 0],
        y=unreachable_points[:, 1],
        z=unreachable_points[:, 2],
        mode='markers',
        name='Unreachable',
        marker=dict(
            size=3,
            color='red',
            opacity=0.2,
            line=dict(width=0)
        ),
        text=[f"X: {x:.1f}<br>Y: {y:.1f}<br>Z: {z:.1f}<br>D: {np.sqrt(x**2+y**2+z**2):.2f}mm" 
              for x, y, z in unreachable_points],
        hovertemplate='<b>Unreachable</b><br>%{text}<extra></extra>'
    ))

fig.update_layout(
    title='Hexapod Leg Workspace (Interactive)',
    scene=dict(
        xaxis_title='X (mm)',
        yaxis_title='Y (mm)',
        zaxis_title='Z (mm)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.3)
        )
    ),
    hovermode='closest',
    height=800,
    width=1200
)

fig.show()

## Z-Slice Analysis - How workspace changes with Z height

In [15]:
# Analyze reachability at different Z heights
z_test_heights = np.arange(Z_MIN, Z_MAX + 20, 5)
reachable_percentages = []
reachable_counts = []

for z in z_test_heights:
    count = 0
    total = 0
    for x in x_range:
        for y in y_range:
            total += 1
            result = inverse_kinematics(x, y, z)
            if result:
                count += 1
    
    percentage = (count / total) * 100 if total > 0 else 0
    reachable_percentages.append(percentage)
    reachable_counts.append(count)

max_z = z_test_heights[np.argmax(reachable_percentages)]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=z_test_heights,
    y=reachable_percentages,
    mode='lines+markers',
    name='Coverage',
    line=dict(color='blue', width=2),
    marker=dict(size=8),
    text=[f"Z: {z:.1f}mm<br>Coverage: {pct:.1f}%<br>Points: {count}" 
          for z, pct, count in zip(z_test_heights, reachable_percentages, reachable_counts)],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    title='Reachable Workspace Coverage vs Z Height',
    xaxis_title='Z height (mm)',
    yaxis_title='Reachable area (%)',
    yaxis=dict(range=[0, 105]),
    height=800,
    width=1200,
    hovermode='closest',
    showlegend=True
)

fig.show()

print("\nReachable area coverage by Z height:")
for z, pct, count in zip(z_test_heights, reachable_percentages, reachable_counts):
    print(f"  Z = {z:7.1f}mm: {pct:6.1f}% ({count} points)")
    
print(f"\nMaximum reachable area of {max(reachable_percentages):.1f}% at Z = {max_z:.1f}mm")


Reachable area coverage by Z height:
  Z =  -300.5mm:    0.0% (0 points)
  Z =  -295.5mm:    1.1% (6 points)
  Z =  -290.5mm:    2.4% (13 points)
  Z =  -285.5mm:    3.7% (20 points)
  Z =  -280.5mm:    4.8% (26 points)
  Z =  -275.5mm:    5.9% (32 points)
  Z =  -270.5mm:    7.0% (38 points)
  Z =  -265.5mm:    7.5% (41 points)
  Z =  -260.5mm:    9.0% (49 points)
  Z =  -255.5mm:    9.7% (53 points)
  Z =  -250.5mm:   11.0% (60 points)
  Z =  -245.5mm:   12.1% (66 points)
  Z =  -240.5mm:   12.7% (69 points)
  Z =  -235.5mm:   14.2% (77 points)
  Z =  -230.5mm:   14.9% (81 points)
  Z =  -225.5mm:   15.6% (85 points)
  Z =  -220.5mm:   16.9% (92 points)
  Z =  -215.5mm:   17.6% (96 points)
  Z =  -210.5mm:   18.4% (100 points)
  Z =  -205.5mm:   19.1% (104 points)
  Z =  -200.5mm:   20.2% (110 points)
  Z =  -195.5mm:   21.3% (116 points)
  Z =  -190.5mm:   21.5% (117 points)
  Z =  -185.5mm:   22.2% (121 points)
  Z =  -180.5mm:   22.4% (122 points)
  Z =  -175.5mm:   24.4% (133 po

In [22]:
# Create XY plane heatmap at Z = max_z with finer resolution
z_slice = max_z

x_range_fine = np.arange(X_MIN, X_MAX + 2, 2.0)  # 5mm step instead of 20mm
y_range_fine = np.arange(Y_MIN, Y_MAX + 2, 2.0)  # 5mm step instead of 20mm

xy_grid = np.zeros((len(y_range_fine), len(x_range_fine)))
xy_text = [[f"X: {x:.0f}<br>Y: {y:.0f}<br>Z: {z_slice:.0f}" 
            for x in x_range_fine] for y in y_range_fine]

print(f"Generating fine-grained heatmap at Z = {z_slice}mm...")
print(f"Grid size: {len(x_range_fine)} x {len(y_range_fine)} = {len(x_range_fine) * len(y_range_fine)} points")

for i, y in enumerate(y_range_fine):
    for j, x in enumerate(x_range_fine):
        result = inverse_kinematics(x, y, z_slice)
        xy_grid[i, j] = 1 if result else 0

fig = go.Figure(data=go.Heatmap(
    z=xy_grid,
    x=x_range_fine,
    y=y_range_fine,
    colorscale='RdYlGn',
    text=xy_text,
    hovertemplate='%{text}<br>Reachable: %{z}<extra></extra>',
    colorbar=dict(
        title="Reachable",
        tickvals=[0, 1],
        ticktext=['No', 'Yes']
    )
))

fig.update_layout(
    title=f'Reachable Workspace at Z = {z_slice:.1f}mm (Fine Resolution: 5mm grid)',
    xaxis_title='X (mm)',
    yaxis_title='Y (mm)',
    height=800,
    width=1200
)

fig.show()
print("Done!")

Generating fine-grained heatmap at Z = -175.53999999999996mm...
Grid size: 152 x 302 = 45904 points


Done!


In [32]:
# Create XY plane heatmap at Z = max_z with finer resolution
z_slice = -100

# Create finer grid for this specific slice
# Create finer grid for this specific slice
x_range_fine = np.arange(X_MIN, X_MAX + 2, 2.0)  # 5mm step instead of 20mm
y_range_fine = np.arange(Y_MIN, Y_MAX + 2, 2.0)  # 5mm step instead of 20mm

xy_grid = np.zeros((len(y_range_fine), len(x_range_fine)))
xy_text = [[f"X: {x:.0f}<br>Y: {y:.0f}<br>Z: {z_slice:.0f}" 
            for x in x_range_fine] for y in y_range_fine]

print(f"Generating fine-grained heatmap at Z = {z_slice}mm...")
print(f"Grid size: {len(x_range_fine)} x {len(y_range_fine)} = {len(x_range_fine) * len(y_range_fine)} points")

for i, y in enumerate(y_range_fine):
    for j, x in enumerate(x_range_fine):
        result = inverse_kinematics(x, y, z_slice)
        xy_grid[i, j] = 1 if result else 0

fig = go.Figure(data=go.Heatmap(
    z=xy_grid,
    x=x_range_fine,
    y=y_range_fine,
    colorscale='RdYlGn',
    text=xy_text,
    hovertemplate='%{text}<br>Reachable: %{z}<extra></extra>',
    colorbar=dict(
        title="Reachable",
        tickvals=[0, 1],
        ticktext=['No', 'Yes']
    )
))

fig.update_layout(
    title=f'Reachable Workspace at Z = {z_slice:.1f}mm (Fine Resolution: 5mm grid)',
    xaxis_title='X (mm)',
    yaxis_title='Y (mm)',
    height=800,
    width=1200
)

fig.show()
print("Done!")

Generating fine-grained heatmap at Z = -100mm...
Grid size: 152 x 302 = 45904 points


Done!


## Generate Workspace Lookup Table for Tripod Gait

Pre-compute the maximum inscribed circle for each Z height to use in the embedded system

In [28]:
# Generate workspace lookup table for tripod gait
# For each Z height, find the largest circle that fits in the reachable workspace
# Assumption: Due to symmetry, center_y = 0 always

# Define Z sampling: 1mm granularity from 0 to -300mm
Z_LOOKUP_MIN = -300.0
Z_LOOKUP_MAX = 0.0
Z_LOOKUP_STEP = 1.0

z_lookup_range = np.arange(Z_LOOKUP_MIN, Z_LOOKUP_MAX + Z_LOOKUP_STEP, Z_LOOKUP_STEP)

workspace_lookup = []

print(f"Generating workspace lookup table for {len(z_lookup_range)} Z heights...")
print(f"Z range: {Z_LOOKUP_MIN} to {Z_LOOKUP_MAX} mm (step: {Z_LOOKUP_STEP} mm)")
print(f"Assumption: center_y = 0 (symmetric configuration)\n")

for idx, z_height in enumerate(z_lookup_range):
    if idx % 50 == 0:
        print(f"  Processing Z = {z_height:.1f} mm ({idx}/{len(z_lookup_range)})...")
    
    # Test potential center_x values (center_y = 0 by symmetry)
    center_y = 0.0
    best_center_x = 0.0
    max_radius = 0.0
    
    # Search for optimal center_x
    # Start from x=0 and go up to MAX_REACH in steps
    for center_x in np.arange(0, MAX_REACH, 2.0):
        
        # Binary search for maximum radius at this center
        min_r, max_r = 0.0, MAX_REACH
        
        for _ in range(15):  # Binary search iterations
            test_r = (min_r + max_r) / 2.0
            
            # Sample points on circle at this radius
            circle_valid = True
            num_samples = 24
            
            for angle_idx in range(num_samples):
                angle = 2 * np.pi * angle_idx / num_samples
                test_x = center_x + test_r * np.cos(angle)
                test_y = center_y + test_r * np.sin(angle)
                
                if not inverse_kinematics(test_x, test_y, z_height):
                    circle_valid = False
                    break
            
            if circle_valid:
                min_r = test_r  # Can go larger
            else:
                max_r = test_r  # Too large
        
        candidate_radius = min_r
        
        if candidate_radius > max_radius:
            max_radius = candidate_radius
            best_center_x = center_x
    
    workspace_lookup.append({
        'z': z_height,
        'center_x': best_center_x,
        'center_y': center_y,
        'radius': max_radius
    })

# Convert to DataFrame for easier viewing
workspace_df = pd.DataFrame(workspace_lookup)

print(f"\nWorkspace lookup table generated: {len(workspace_lookup)} entries")
print(f"\nSample entries:")
print(workspace_df.head(10))
print(f"\nStatistics:")
print(f"  Max radius: {workspace_df['radius'].max():.2f} mm at Z = {workspace_df.loc[workspace_df['radius'].idxmax(), 'z']:.1f} mm")
print(f"  Min radius (non-zero): {workspace_df[workspace_df['radius'] > 0]['radius'].min():.2f} mm")
print(f"  Mean radius: {workspace_df['radius'].mean():.2f} mm")

# Check specific Z heights
for z_check in [-50.0, -100.0, -150.0, -200.0]:
    if z_check in workspace_df['z'].values:
        entry = workspace_df[workspace_df['z'] == z_check].iloc[0]
        print(f"\nAt Z = {z_check}mm:")
        print(f"  Center: ({entry['center_x']:.1f}, {entry['center_y']:.1f})")
        print(f"  Radius: {entry['radius']:.1f} mm")

Generating workspace lookup table for 301 Z heights...
Z range: -300.0 to 0.0 mm (step: 1.0 mm)
Assumption: center_y = 0 (symmetric configuration)

  Processing Z = -300.0 mm (0/301)...
  Processing Z = -250.0 mm (50/301)...
  Processing Z = -250.0 mm (50/301)...
  Processing Z = -200.0 mm (100/301)...
  Processing Z = -200.0 mm (100/301)...
  Processing Z = -150.0 mm (150/301)...
  Processing Z = -150.0 mm (150/301)...
  Processing Z = -100.0 mm (200/301)...
  Processing Z = -100.0 mm (200/301)...
  Processing Z = -50.0 mm (250/301)...
  Processing Z = -50.0 mm (250/301)...
  Processing Z = 0.0 mm (300/301)...

Workspace lookup table generated: 301 entries

Sample entries:
       z  center_x  center_y     radius
0 -300.0      14.0       0.0   3.237629
1 -299.0      20.0       0.0   9.153409
2 -298.0      24.0       0.0  13.069748
3 -297.0      28.0       0.0  16.976915
4 -296.0      32.0       0.0  20.040280
5 -295.0      34.0       0.0  22.800978
6 -294.0      36.0       0.0  24.7178

In [31]:
# Generate C++ header file with workspace lookup table
cpp_header = """// WorkspaceLookup.h
// Auto-generated workspace lookup table for tripod gait
// Generated from workspace_checker.ipynb

#ifndef WORKSPACE_LOOKUP_H
#define WORKSPACE_LOOKUP_H

#include <Arduino.h>

// Workspace data structure
struct WorkspaceEntry {
    float z;          // Z height in mm
    float center_x;   // Center X coordinate in mm
    float center_y;   // Center Y coordinate in mm  
    float radius;     // Maximum safe radius in mm
};

// Lookup table constants
constexpr int WORKSPACE_TABLE_SIZE = """ + str(len(workspace_lookup)) + """;
constexpr float WORKSPACE_Z_MIN = """ + f"{Z_LOOKUP_MIN:.1f}" + """f;
constexpr float WORKSPACE_Z_MAX = """ + f"{Z_LOOKUP_MAX:.1f}" + """f;
constexpr float WORKSPACE_Z_STEP = """ + f"{Z_LOOKUP_STEP:.1f}" + """f;

// Workspace lookup table
const WorkspaceEntry WORKSPACE_TABLE[WORKSPACE_TABLE_SIZE] PROGMEM = {
"""

# Add data entries
for entry in workspace_lookup:
    cpp_header += f"    {{ {entry['z']:.1f}f, {entry['center_x']:.2f}f, {entry['center_y']:.2f}f, {entry['radius']:.2f}f }},\n"

cpp_header += """};

// Helper function to find workspace parameters for a given Z height
inline bool getWorkspaceAtZ(float z, float& center_x, float& center_y, float& radius) {
    // Clamp Z to valid range
    if (z < WORKSPACE_Z_MIN || z > WORKSPACE_Z_MAX) {
        return false;
    }
    
    // Find nearest Z entry (round to nearest mm)
    int index = int((z - WORKSPACE_Z_MIN) / WORKSPACE_Z_STEP + 0.5f);
    index = constrain(index, 0, WORKSPACE_TABLE_SIZE - 1);
    
    // Read from PROGMEM
    WorkspaceEntry entry;
    memcpy_P(&entry, &WORKSPACE_TABLE[index], sizeof(WorkspaceEntry));
    
    center_x = entry.center_x;
    center_y = entry.center_y;
    radius = entry.radius;
    
    return (radius > 0.0f);  // Return false if no workspace at this height
}

#endif // WORKSPACE_LOOKUP_H
"""

# Write to file
with open('include/WorkspaceLookup.h', 'w') as f:
    f.write(cpp_header)

print("Generated include/WorkspaceLookup.h")
print(f"Table size: {len(workspace_lookup)} entries")
print(f"Estimated memory: {len(workspace_lookup) * 16} bytes (stored in PROGMEM)")

Generated include/WorkspaceLookup.h
Table size: 301 entries
Estimated memory: 4816 bytes (stored in PROGMEM)
